# Notebook 04: previsões no teste fixo e explicabilidade (critério 10)

Último notebook do ciclo. Ele carrega o **modelo vigente** a partir do manifesto, gera previsões no **teste fixo**, calcula as métricas finais e explica as decisões do modelo com **SHAP**. No fim, registra o resumo do ciclo e confirma a ingestão do lote.

| | Arquivos |
|---|---|
| **Lê** | `config/config.yaml`, `artifacts/ciclo_atual.json`, `models/modelo_vigente.json`, `models/historico/<modelo>.joblib`, `data/raw/teste_fixo.csv`, `data/processed/treino_<data_version>.csv` (só como fundo do SHAP linear) |
| **Grava** | `reports/figuras/<cycle_id>_04_*.csv` e `.png`, `reports/figuras/comparacao_ciclos.csv` e `.png`, `models/estado_ingestao.json`, `artifacts/ciclo_atual.json`, um run de previsões no MLflow |

**O teste fixo só é usado aqui, e só para medir.** Os 905 e-mails do teste foram separados uma única vez, antes de tudo, e nenhum ciclo os usa para escolher features, comparar modelos ou decidir promoção. Por isso as métricas abaixo são uma estimativa honesta do desempenho em e-mails novos. Como o teste é sempre o mesmo, os resultados de ciclos diferentes são comparáveis entre si.

## 0. Preparação do ambiente

Esta célula faz o mesmo notebook rodar sem alterações no **Google Colab**, no **GitHub Actions** e **localmente**:

* **No Colab** (`'google.colab' in sys.modules`): clona o repositório definido em `REPO_URL` dentro de `/content`, entra na pasta e instala o `requirements.txt`. Se o repositório já estiver clonado, só atualiza com `git pull`. Se a instalação trocar a versão de uma biblioteca que o Colab já tinha carregado, a sessão é reiniciada automaticamente: basta executar esta célula de novo.
* **Fora do Colab**: procura a raiz do repositório (a pasta que contém `config/config.yaml`) subindo a partir da pasta atual e muda o diretório de trabalho para ela.

Nos dois casos a raiz entra no `sys.path`, o que permite `from src... import ...`. Todos os caminhos usados depois são relativos à raiz.

**`REPO_URL` já aponta para o repositório do projeto (`https://github.com/eduardaramalho/n3-analise-preditiva.git`). Só altere se o repositório mudar de endereço.**

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/eduardaramalho/n3-analise-preditiva.git"  # repositório do projeto
REPO_BRANCH = "main"

NA_COLAB = "google.colab" in sys.modules

if NA_COLAB:
    pasta_repo = Path("/content") / Path(REPO_URL).stem
    if not (pasta_repo / ".git").exists():
        subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, str(pasta_repo)], check=True)
    else:
        subprocess.run(["git", "-C", str(pasta_repo), "pull", "--ff-only"], check=False)
    os.chdir(pasta_repo)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    # Se o pip trocou a versão de uma biblioteca já carregada, a sessão precisa reiniciar
    from importlib.metadata import version
    trocadas = [m for m, dist in {"numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn",
                                  "matplotlib": "matplotlib"}.items()
                if m in sys.modules and getattr(sys.modules[m], "__version__", version(dist)) != version(dist)]
    if trocadas:
        print(f"Versões atualizadas ({', '.join(trocadas)}). Reiniciando a sessão: execute esta célula de novo.")
        os.kill(os.getpid(), 9)
else:
    raiz = Path.cwd().resolve()
    while not (raiz / "config" / "config.yaml").exists():
        if raiz.parent == raiz:
            raise FileNotFoundError("Raiz do repositório não encontrada (config/config.yaml).")
        raiz = raiz.parent
    os.chdir(raiz)

RAIZ = Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
print("Ambiente:", "Google Colab" if NA_COLAB else "local / GitHub Actions")
print("Raiz do repositório:", RAIZ)

## 1. Configuração e modelo vigente

O modelo é carregado do arquivo indicado em `models/modelo_vigente.json` (campo `arquivo_modelo`, em `models/historico/`). Se o arquivo não existir, a função usa a cópia do MLflow (`modelo_uri`).

Se nenhum modelo foi promovido até agora (F1 abaixo do mínimo no primeiro ciclo), não há o que avaliar: o notebook pula as previsões, mas ainda registra o ciclo.

In [ ]:
import os
import warnings

warnings.filterwarnings("ignore", message="IProgress not found")
import mlflow
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

from src.utils import carregar_config, caminho, fixar_seed
from src import ingestao, modelagem as M, promocao as P, explicabilidade as E, graficos
from src.preprocessamento import colunas_features
from src.rastreamento import configurar_mlflow

cfg = carregar_config()
SEED = cfg["projeto"]["seed"]
fixar_seed(SEED)
graficos.aplicar_estilo()

ciclo = ingestao.carregar_ciclo(cfg)
CYCLE_ID, DATA_VERSION = ciclo["cycle_id"], ciclo["data_version"]
if ciclo.get("etapas", {}).get("03_competicao_promocao") != "concluido":
    raise RuntimeError("O notebook 03 não foi concluído neste ciclo.")
ALVO, COL_ID = cfg["dados"]["alvo"], cfg["dados"]["coluna_id"]

VIGENTE = P.carregar_vigente(cfg)
TEM_VIGENTE = VIGENTE is not None
if TEM_VIGENTE:
    modelo = P.carregar_modelo_vigente(cfg, VIGENTE)
    print(f"Vigente: {VIGENTE['algoritmo']} | run {VIGENTE['run_id']}")
    print(f"Promovido no ciclo {VIGENTE['cycle_id']} com {VIGENTE['data_version']}")
    print(f"F1 na validação cruzada do ciclo de origem: {VIGENTE['metrica']['valor']:.4f}")
    print("Decisão deste ciclo:", VIGENTE["ultima_decisao"]["decisao"], "|", VIGENTE["ultima_decisao"]["motivo"])
    print(modelo)
else:
    print("Nenhum modelo vigente. Previsões e SHAP serão pulados neste ciclo.")

## 2. Previsões no teste fixo

O Pipeline recebe as **57 colunas brutas**: ele mesmo seleciona as features eleitas, padroniza (se for o caso) e prevê. Para cada e-mail temos a classe prevista (limiar padrão de 0,5) e a probabilidade de ser spam.

In [ ]:
teste = pd.read_csv(caminho(cfg["dados"]["teste_fixo"]))
X_teste, y_teste = teste[colunas_features(teste, cfg)], teste[ALVO]
print(f"Teste fixo: {len(teste)} e-mails | spam = {y_teste.mean():.1%}")

if TEM_VIGENTE:
    y_pred = modelo.predict(X_teste)
    prob_spam = modelo.predict_proba(X_teste)[:, 1]
    METRICAS_TESTE = M.calcular_metricas(y_teste, y_pred, prob_spam)
    tabela_metricas = pd.DataFrame({
        "teste_fixo": METRICAS_TESTE,
        "validacao_cruzada_origem": {k.replace("_media", ""): v for k, v in VIGENTE["metrica"]["todas"].items()},
    })
    tabela_metricas["diferenca"] = tabela_metricas["teste_fixo"] - tabela_metricas["validacao_cruzada_origem"]
    graficos.salvar_tabela(tabela_metricas.round(4), cfg, CYCLE_ID, "04_metricas_teste", index=True)
    display(tabela_metricas.round(4))

**Como ler:** a coluna `validacao_cruzada_origem` é o desempenho estimado no ciclo em que o vigente foi promovido. Se o `teste_fixo` ficar próximo dela (diferença de poucos centésimos), a validação cruzada foi uma boa estimativa e não houve sobreajuste nem vazamento. Uma queda grande no teste seria um sinal de alerta.

## 3. Matriz de confusão e relatório por classe

* **Verdadeiro positivo:** spam detectado. **Verdadeiro negativo:** e-mail legítimo liberado.
* **Falso positivo:** e-mail legítimo marcado como spam. É o erro mais caro para o usuário, porque ele pode perder uma mensagem importante. A documentação do Spambase destaca esse ponto.
* **Falso negativo:** spam que chegou na caixa de entrada. Incomoda, mas é menos grave.

O percentual em cada célula é sobre a linha (classe real): na linha "não spam", o valor em "spam" é a taxa de falsos positivos.

In [ ]:
if TEM_VIGENTE:
    matriz = confusion_matrix(y_teste, y_pred, labels=[0, 1])
    fig = graficos.grafico_matriz_confusao(matriz, f"Matriz de confusão no teste fixo ({VIGENTE['algoritmo']})")
    graficos.salvar_figura(fig, cfg, CYCLE_ID, "04_matriz_confusao")
    tabela_matriz = pd.DataFrame(matriz, index=["real_nao_spam", "real_spam"], columns=["previsto_nao_spam", "previsto_spam"])
    graficos.salvar_tabela(tabela_matriz, cfg, CYCLE_ID, "04_matriz_confusao", index=True)
    relatorio = pd.DataFrame(classification_report(y_teste, y_pred, target_names=["não spam", "spam"], output_dict=True)).T
    graficos.salvar_tabela(relatorio.round(4), cfg, CYCLE_ID, "04_relatorio_classificacao", index=True)
    vn, fp, fn, vp = matriz.ravel()
    print(f"Falsos positivos: {fp} de {vn + fp} e-mails legítimos ({fp / (vn + fp):.1%})")
    print(f"Falsos negativos: {fn} de {fn + vp} spams ({fn / (fn + vp):.1%})")
    display(relatorio.round(4))

## 4. Exemplos de previsão

Uma amostra de e-mails do teste com a classe real, a prevista e a probabilidade de spam: os 5 mais claramente spam, os 5 mais claramente legítimos, os erros e os 5 casos mais incertos (probabilidade perto de 0,5). Os casos incertos são os que um sistema real poderia mandar para revisão.

In [ ]:
if TEM_VIGENTE:
    resultado = pd.DataFrame({COL_ID: teste[COL_ID], "classe_real": y_teste, "classe_prevista": y_pred,
                              "prob_spam": prob_spam.round(4)})
    resultado["acertou"] = resultado["classe_real"] == resultado["classe_prevista"]
    resultado["tipo"] = np.select(
        [resultado.acertou & (resultado.classe_real == 1), resultado.acertou & (resultado.classe_real == 0),
         resultado.classe_real == 0], ["VP", "VN", "FP"], default="FN")
    incertos = resultado.iloc[np.argsort(np.abs(prob_spam - 0.5))[:5]]
    exemplos = pd.concat([
        resultado.nlargest(5, "prob_spam").assign(grupo="mais provável spam"),
        resultado.nsmallest(5, "prob_spam").assign(grupo="mais provável não spam"),
        resultado[~resultado.acertou].assign(grupo="erro"),
        incertos.assign(grupo="mais incerto"),
    ]).drop_duplicates(subset=COL_ID)
    graficos.salvar_tabela(resultado, cfg, CYCLE_ID, "04_previsoes_teste")
    graficos.salvar_tabela(exemplos, cfg, CYCLE_ID, "04_exemplos_previsao")
    display(exemplos.groupby("grupo", sort=False).head(5))

## 5. Explicabilidade com SHAP (critério 10)

**O que é SHAP.** Para cada e-mail, o SHAP reparte a previsão entre as features. Ele parte de um **valor base** (a previsão média) e soma a contribuição de cada feature: valores positivos empurram para spam, negativos para não spam. A soma do valor base com todas as contribuições dá a previsão daquele e-mail. A ideia vem dos valores de Shapley, da teoria dos jogos: cada feature recebe a sua contribuição média considerando todas as combinações possíveis com as outras.

**Qual explainer.** Depende do algoritmo do vigente, lido do manifesto:

* **Random Forest:** `TreeExplainer`, que calcula os valores exatos percorrendo as árvores. Os valores estão em **probabilidade de spam** (ex.: +0,15 = a feature aumenta em 15 pontos percentuais a chance de spam).
* **Regressão Logística:** `LinearExplainer`. Os valores estão em **log-odds** (a escala interna da regressão logística). O fundo (dados de referência para o valor base) é uma amostra do **treino**, nunca do teste.

**Parâmetros (config.yaml, seção `explicabilidade`):** o gráfico global usa uma amostra estratificada de `n_amostras_explicar` = 300 e-mails do teste, o que basta para estabilizar as médias e mantém a execução rápida. O fundo do `LinearExplainer` usa `n_amostras_fundo` = 200 e-mails do treino.

Os gráficos mostram os valores **originais** das features (antes do `StandardScaler`), para que a leitura faça sentido.

In [ ]:
if TEM_VIGENTE:
    ex = cfg["explicabilidade"]
    treino = pd.read_csv(caminho(ciclo["arquivo_treino"]))
    X_fundo = treino[colunas_features(treino, cfg)].sample(n=min(ex["n_amostras_fundo"], len(treino)), random_state=SEED)
    n_explicar = min(ex["n_amostras_explicar"], len(teste))
    amostra = teste if n_explicar == len(teste) else train_test_split(
        teste, train_size=n_explicar, stratify=teste[ALVO], random_state=SEED)[0]
    X_amostra = amostra[colunas_features(teste, cfg)]
    exp_global, NOME_EXPLAINER = E.explicar(modelo, X_amostra, X_fundo, SEED)
    ESCALA = "probabilidade de spam" if "Tree" in NOME_EXPLAINER else "log-odds de spam"
    importancia = E.importancia_global(exp_global)
    graficos.salvar_tabela(importancia, cfg, CYCLE_ID, "04_shap_importancia_global")
    print(f"Explainer: {NOME_EXPLAINER} | {len(X_amostra)} e-mails explicados | valor base = {exp_global.base_values[0]:.4f}")
    display(importancia)

### 5.1 Importância global

**Barras:** a média do valor absoluto do SHAP de cada feature, ou seja, o quanto ela move a previsão em média, para qualquer lado. A cor indica para que lado ela empurra na média.

**Beeswarm:** cada ponto é um e-mail. A posição horizontal é o valor SHAP (direita = empurra para spam) e a cor é o valor da feature (vermelho = alto, azul = baixo). Exemplo de leitura: se em `char_freq_$` os pontos vermelhos ficam à direita, então muitos "$" no e-mail aumentam a chance de spam. Em `word_freq_george` e `word_freq_hp` acontece o contrário: o dataset foi montado com e-mails de um funcionário da HP chamado George, então essas palavras indicam e-mail legítimo, como alerta a própria documentação do dataset.

In [ ]:
if TEM_VIGENTE:
    fig = graficos.grafico_shap_global(importancia, f"Importância global SHAP ({VIGENTE['algoritmo']})",
                                       f"média de |SHAP| ({ESCALA})")
    graficos.salvar_figura(fig, cfg, CYCLE_ID, "04_shap_importancia_global")
    plt.figure()
    shap.plots.beeswarm(exp_global, max_display=len(importancia), show=False)
    fig = plt.gcf()
    fig.set_size_inches(8, 0.32 * len(importancia) + 2)
    plt.title(f"SHAP beeswarm ({ESCALA})", loc="left", fontweight="bold")
    plt.tight_layout()
    graficos.salvar_figura(fig, cfg, CYCLE_ID, "04_shap_beeswarm")
    plt.show()

### 5.2 Explicação de e-mails individuais

Três e-mails do teste escolhidos automaticamente: o spam que o modelo identificou com mais confiança, o e-mail legítimo identificado com mais confiança e o erro em que o modelo estava mais confiante (ou, se não houver erro, o caso mais incerto).

**Gráfico waterfall:** começa no valor base (embaixo, `E[f(X)]`) e cada barra soma a contribuição de uma feature, até chegar na previsão do e-mail (em cima, `f(x)`). Barras vermelhas empurram para spam, azuis para não spam. Ao lado do nome aparece o valor da feature naquele e-mail.

In [ ]:
if TEM_VIGENTE:
    escolhidos = E.escolher_exemplos(y_teste, y_pred, prob_spam, n=ex["n_exemplos_individuais"])
    X_escolhidos = X_teste.iloc[[pos for _, pos in escolhidos]]
    exp_ind, _ = E.explicar(modelo, X_escolhidos, X_fundo, SEED)
    linhas_ind = []
    for k, (descricao, pos) in enumerate(escolhidos):
        id_ = int(teste.iloc[pos][COL_ID])
        plt.figure()
        shap.plots.waterfall(exp_ind[k], max_display=10, show=False)
        fig = plt.gcf()
        fig.set_size_inches(8, 5)
        plt.title(f"E-mail {id_}: {descricao}\nreal = {int(y_teste.iloc[pos])}, previsto = {int(y_pred[pos])}, "
                  f"prob. spam = {prob_spam[pos]:.3f}", loc="left", fontsize=10)
        plt.tight_layout()
        graficos.salvar_figura(fig, cfg, CYCLE_ID, f"04_shap_email_{k + 1}_{id_}")
        plt.show()
        contrib = pd.DataFrame({"feature": exp_ind.feature_names, "valor_feature": exp_ind.data[k],
                                "shap": exp_ind.values[k]}).sort_values("shap", key=np.abs, ascending=False)
        for _, r in contrib.head(5).iterrows():
            linhas_ind.append({"email": id_, "caso": descricao, "classe_real": int(y_teste.iloc[pos]),
                               "classe_prevista": int(y_pred[pos]), "prob_spam": round(float(prob_spam[pos]), 4),
                               "valor_base": round(float(exp_ind.base_values[k]), 4), **r.to_dict()})
    tabela_ind = pd.DataFrame(linhas_ind)
    graficos.salvar_tabela(tabela_ind.round(4), cfg, CYCLE_ID, "04_shap_emails_individuais")
    display(tabela_ind.round(4))

## 6. Registro no MLflow

Um run de **previsões** por ciclo, com as métricas no teste fixo, a matriz de confusão, os exemplos e todos os gráficos SHAP. A tag `run_id_vigente` liga este run ao run do modelo que gerou as previsões.

In [ ]:
configurar_mlflow(cfg)
EVENTO = os.environ.get("GITHUB_EVENT_NAME", "colab" if "google.colab" in __import__("sys").modules else "local")
with mlflow.start_run(run_name=f"previsoes_{CYCLE_ID}") as run:
    mlflow.set_tags({"etapa": "previsoes", "cycle_id": CYCLE_ID, "data_version": DATA_VERSION, "evento": EVENTO,
                     "run_id_vigente": VIGENTE["run_id"] if TEM_VIGENTE else "nenhum"})
    mlflow.log_params({"cycle_id": CYCLE_ID, "data_version": DATA_VERSION, "n_teste": len(teste), "seed": SEED})
    if TEM_VIGENTE:
        mlflow.log_params({"algoritmo_vigente": VIGENTE["algoritmo"], "explainer": NOME_EXPLAINER,
                           "data_version_vigente": VIGENTE["data_version"]})
        mlflow.log_metrics({f"teste_{k}": float(v) for k, v in METRICAS_TESTE.items()})
        for arq in sorted(caminho(cfg["caminhos"]["figuras"]).glob(f"{CYCLE_ID}_04_*")):
            destino = "shap" if "_shap_" in arq.name else "previsoes"
            mlflow.log_artifact(str(arq), artifact_path=destino)
    RUN_ID_PREVISOES = run.info.run_id
print("Run de previsões:", RUN_ID_PREVISOES, "| evento:", EVENTO)

## 7. Fechamento do ciclo

Três ações finais, executadas só se todos os notebooks anteriores chegaram até aqui:

1. **`reports/figuras/comparacao_ciclos.csv`** ganha uma linha com o resumo do ciclo (lote novo, features eleitas, F1 de A e B, vencedor, F1 do vigente reavaliado, decisão, métricas no teste e o evento que disparou o ciclo). O gráfico `comparacao_ciclos.png` mostra a evolução.
2. **`models/estado_ingestao.json`** passa a marcar os lotes deste ciclo como ingeridos. Se algo tivesse falhado antes, o lote não seria confirmado e o próximo ciclo tentaria de novo.
3. **`artifacts/ciclo_atual.json`** registra o notebook 04 como concluído.

In [ ]:
d = ciclo["decisao"]
linha = {
    "cycle_id": CYCLE_ID, "data_version": DATA_VERSION, "evento": EVENTO,
    "lote_novo": ciclo["lote_novo"], "lotes_no_treino": "+".join(ciclo["lotes_ciclo"]),
    "linhas_treino": ciclo["linhas_treino"], "n_features_eleitas": len(ciclo["features_eleitas"]),
}
# F1 de A e B vêm dos runs da competição
cliente = mlflow.MlflowClient()
for chave, run_id in ciclo["run_ids_modelos"].items():
    linha[f"f1_cv_{chave}"] = cliente.get_run(run_id).data.metrics["f1_media"]
linha.update({
    "vencedor": ciclo["vencedor"], "algoritmo_vencedor": d["algoritmo_vencedor"],
    "f1_vigente_reavaliado": d["f1_vigente_reavaliado"], "ganho_f1": d["ganho_f1"],
    "decisao": d["decisao"], "motivo": d["motivo"],
    "algoritmo_vigente": VIGENTE["algoritmo"] if TEM_VIGENTE else None,
    "run_id_vigente": VIGENTE["run_id"] if TEM_VIGENTE else None,
    **({f"teste_{k}": round(float(v), 4) for k, v in METRICAS_TESTE.items()} if TEM_VIGENTE else {}),
    "run_id_decisao": ciclo["run_id_decisao"], "run_id_previsoes": RUN_ID_PREVISOES,
})
historico_ciclos = P.registrar_comparacao_ciclos(cfg, linha)
fig = graficos.grafico_evolucao_ciclos(historico_ciclos)
fig.savefig(caminho(cfg["caminhos"]["figuras"]) / "comparacao_ciclos.png")

estado = ingestao.confirmar_ingestao(cfg, ciclo)
ingestao.atualizar_ciclo(cfg, etapas={**ciclo["etapas"], "04_previsoes": "concluido"}, run_id_previsoes=RUN_ID_PREVISOES)
print("Lotes ingeridos:", estado["lotes_ingeridos"])
historico_ciclos

## 8. Resumo

| Arquivo em `reports/figuras/` | Conteúdo |
|---|---|
| `<cycle_id>_04_metricas_teste.csv` | métricas no teste fixo x validação cruzada |
| `<cycle_id>_04_matriz_confusao.png` / `.csv` e `_04_relatorio_classificacao.csv` | erros por classe |
| `<cycle_id>_04_previsoes_teste.csv` e `_04_exemplos_previsao.csv` | previsões e exemplos |
| `<cycle_id>_04_shap_importancia_global.png` / `.csv` e `_04_shap_beeswarm.png` | explicação global |
| `<cycle_id>_04_shap_email_<n>_<id>.png` e `_04_shap_emails_individuais.csv` | explicação de e-mails individuais |
| `comparacao_ciclos.csv` / `.png` | resumo acumulado de todos os ciclos |

Fim do ciclo. O próximo ciclo começa de novo pelo notebook 01.

In [ ]:
if TEM_VIGENTE:
    print(f"Ciclo {CYCLE_ID} concluído | vigente {VIGENTE['algoritmo']} | F1 no teste fixo = {METRICAS_TESTE['f1']:.4f}")
else:
    print(f"Ciclo {CYCLE_ID} concluído sem modelo vigente")